# Building Simple ReAct Agent from Scratch In Python

Requires a locally configured OPENAI_API_KEY; MODEL defaults to gpt-5.4-mini. API calls incur charges. Weather/forecast tools are fixed fictional teaching data, not live observations. The short visible Thought field describes an operation and does not expose private model reasoning. Run cells in order.

```bash
python -m pip install openai==2.43.0 python-dotenv==1.2.4
```

In [ ]:
import os, re, math, json
from copy import deepcopy
from importlib.metadata import version
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
MODEL = os.getenv('MODEL', 'gpt-5.4-mini')
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Set OPENAI_API_KEY locally; never paste it into the notebook.')
client = OpenAI(timeout=30, max_retries=0)
print('OpenAI SDK:', version('openai'), 'Model:', MODEL)

In [ ]:
class Agent:
    def __init__(self, system='', completion_client=None):
        self.client = completion_client or client
        self.messages = [{'role': 'developer', 'content': system}] if system else []

    def __call__(self, message):
        if not isinstance(message, str) or not message.strip():
            raise ValueError('Expected a nonempty message')
        candidate = deepcopy(self.messages) + [{'role': 'user', 'content': message}]
        result = self.execute(candidate)
        # Commit both messages only after a complete valid provider response.
        self.messages = candidate + [{'role': 'assistant', 'content': result}]
        return result

    def execute(self, messages):
        completion = self.client.chat.completions.create(
            model=MODEL, messages=messages, temperature=0, reasoning_effort='none',
            max_completion_tokens=512, store=False)
        if len(completion.choices) != 1:
            raise RuntimeError('Expected one completion')
        choice = completion.choices[0]
        if choice.finish_reason != 'stop' or choice.message.refusal:
            raise RuntimeError('Incomplete or refused completion')
        result = choice.message.content
        if not isinstance(result, str) or not result.strip():
            raise RuntimeError('Empty model response')
        return result.strip()
print('Agent stages history and commits complete user/assistant pairs.')

In [ ]:
prompt = """
You are a small teaching ReAct agent. Weather tools return FIXED FICTIONAL
fixtures, not live weather. Include the exact word fixture in every weather or
forecast Answer, as well as the observed weather/forecast values.
You alternate a registered action with the observation returned by the caller.
For an action use EXACTLY:
Thought: a brief one-line description of the next operation (not private reasoning)
Action: action_name: arguments
PAUSE
Use one action per response. Never invent an Observation. Once enough actual
observations have arrived, return only Answer: followed by the result.

Available actions:
get_current_weather: Helsinki
Returns a fixed 32 F weather fixture for the requested city.
get_forecast: Helsinki, 1
Returns a fixed sunny/mild fixture; integer days must be 1 through 7.
convert_temperature: 32 F to C
Converts a finite numeric temperature between F and C.
When a question asks for Celsius and weather is Fahrenheit, call the conversion
tool after receiving the weather observation, even if the arithmetic seems easy.

Example:
Question: What is the fixture temperature in Paris in Celsius?
Thought: Look up the fixed temperature.
Action: get_current_weather: Paris
PAUSE
Observation: Fixture: the temperature in Paris is 32 F.
Thought: Convert the observed value.
Action: convert_temperature: 32 F to C
PAUSE
Observation: 32.0 F is 0.0 C.
Answer: The fixed Paris fixture is 0.0 C, not live weather.
""".strip()
print(prompt)

In [ ]:
def valid_city(location):
    if not isinstance(location, str) or not re.fullmatch(r"[A-Za-z][A-Za-z .'-]{0,60}", location):
        raise ValueError('Expected a short city name')
    return location.strip()

def get_current_weather(location):
    return f'Fixture: the temperature in {valid_city(location)} is 32 F.'

def get_forecast(location, days):
    if type(days) is not int or not 1 <= days <= 7:
        raise ValueError('Forecast days must be an integer from 1 to 7')
    return f'Fixture: {valid_city(location)} has sunny, mild weather for {days} day(s).'

def convert_temperature(conversion):
    match = re.fullmatch(r'([+-]?(?:[0-9]+(?:\.[0-9]*)?|\.[0-9]+))\s+([FC])\s+to\s+([FC])', conversion, re.I)
    if not match:
        raise ValueError('Use a finite temperature and F to C or C to F')
    value, source, target = float(match[1]), match[2].upper(), match[3].upper()
    if not math.isfinite(value) or abs(value) > 1e6 or source == target:
        raise ValueError('Unsupported temperature conversion')
    result = (value - 32) * 5 / 9 if source == 'F' else value * 9 / 5 + 32
    return f'{value:.1f} {source} is {result:.1f} {target}.'

known_actions = {'get_current_weather': get_current_weather,
                 'get_forecast': get_forecast, 'convert_temperature': convert_temperature}

def parse_response(text):
    if not isinstance(text, str):
        raise ValueError('Expected a response string')
    if re.fullmatch(r'Answer: [^\n]+', text) and 'Action:' not in text:
        return ('answer', text[8:])
    match = re.fullmatch(r'Thought: [^\n]+\nAction: ([a-z_]+): ([^\n]+)\nPAUSE', text)
    if not match or match[1] not in known_actions:
        raise ValueError('Malformed or unknown action')
    return ('action', match[1], match[2])

def dispatch(name, argument):
    if name not in known_actions:
        raise ValueError('Unknown action')
    if name == 'get_forecast':
        match = re.fullmatch(r'(.+), ([1-7])', argument)
        if not match:
            raise ValueError('Forecast needs city, days')
        return get_forecast(match[1], int(match[2]))
    return known_actions[name](argument)
print(convert_temperature('32 F to C'))
print(convert_temperature('100 C to F'))

In [ ]:
abot = Agent(prompt)
first = abot('What is the temperature in Helsinki in Celsius?')
print(first)
action = parse_response(first)
assert action[:2] == ('action', 'get_current_weather')
weather_observation = dispatch(action[1], action[2])
print('Observation:', weather_observation)
second = abot('Observation: ' + weather_observation)
print(second)
conversion_action = parse_response(second)
assert conversion_action[:2] == ('action', 'convert_temperature')
conversion_observation = dispatch(conversion_action[1], conversion_action[2])
assert conversion_observation == '32.0 F is 0.0 C.'
print('Observation:', conversion_observation)
answer = abot('Observation: ' + conversion_observation)
assert parse_response(answer)[0] == 'answer'
assert re.search(r'0(?:\.0)?\s*°?\s*C', answer) and 'fixture' in answer.lower()
print(answer)

In [ ]:
# Keep this separate from the current-temperature observation.
forecast_observation = get_forecast('Helsinki', 1)
print(forecast_observation)
assert 'sunny' in forecast_observation and 'Fixture:' in forecast_observation

In [ ]:
print(json.dumps(abot.messages, indent=2, ensure_ascii=False))
assert [m['role'] for m in abot.messages] == ['developer', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant']
assert weather_observation in abot.messages[3]['content']
assert conversion_observation in abot.messages[5]['content']
assert not any(forecast_observation in m['content'] for m in abot.messages)

In [ ]:
def query(question, max_turns=6, agent=None):
    if type(max_turns) is not int or not 1 <= max_turns <= 8:
        raise ValueError('max_turns must be from 1 to 8')
    agent = agent or Agent(prompt)
    initial = deepcopy(agent.messages)
    next_prompt = question
    events = []
    try:
        for _ in range(max_turns):
            result = agent(next_prompt)
            parsed = parse_response(result)
            print(result)
            if parsed[0] == 'answer':
                return parsed[1], events, agent
            observation = dispatch(parsed[1], parsed[2])
            events.append({'action': parsed[1], 'argument': parsed[2], 'observation': observation})
            print('Observation:', observation)
            next_prompt = 'Observation: ' + observation
        raise RuntimeError('Agent turn limit reached')
    except Exception:
        agent.messages = initial
        raise

automated_answer, events, automated_agent = query('What is the temperature in Helsinki in Celsius?')
assert [e['action'] for e in events] == ['get_current_weather', 'convert_temperature']
assert events[-1]['observation'] == '32.0 F is 0.0 C.'
assert re.search(r'0(?:\.0)?\s*°?\s*C', automated_answer) and 'fixture' in automated_answer.lower()
forecast_answer, forecast_events, forecast_agent = query('Give the fixed Helsinki forecast for 3 days.')
assert [e['action'] for e in forecast_events] == ['get_forecast']
assert forecast_events[0]['argument'] == 'Helsinki, 3'
assert 'sunny' in forecast_answer.lower() and 'fixture' in forecast_answer.lower()
assert not any('3 days' in m['content'] for m in automated_agent.messages)

In [ ]:
from types import SimpleNamespace
def rejected(function, *args):
    try:
        function(*args)
    except (ValueError, RuntimeError):
        return
    raise AssertionError('Invalid input was accepted')
for value in ['nan F to C', 'inf C to F', '1 K to C', '10 F to F', '1 F into C', '1e999 F to C']:
    rejected(convert_temperature, value)
for text in ['Action: delete: anything', 'Answer: ok\nAction: get_current_weather: Helsinki', 'Thought: ok\nAction: unknown: Helsinki\nPAUSE']:
    rejected(parse_response, text)
rejected(dispatch, 'get_forecast', 'Helsinki, 0')
rejected(dispatch, 'get_current_weather', '../secret')
assert convert_temperature('-40 C to F') == '-40.0 C is -40.0 F.'
assert convert_temperature('212 F to C') == '212.0 F is 100.0 C.'

# Offline fault injection exercises failures; it does not replace the live calls above.
class FailedCompletions:
    def create(self, **kwargs):
        raise RuntimeError('Deliberate offline provider failure')
faulty = Agent(prompt, SimpleNamespace(chat=SimpleNamespace(completions=FailedCompletions())))
before = deepcopy(faulty.messages)
rejected(faulty, 'hello')
assert faulty.messages == before
class ScriptedAgent:
    def __init__(self, response):
        self.response, self.messages = response, []
    def __call__(self, message):
        self.messages.extend([{'role': 'user', 'content': message}, {'role': 'assistant', 'content': self.response}])
        return self.response
for response in ['garbage', 'Thought: next\nAction: unknown: Helsinki\nPAUSE',
                 'Thought: repeat\nAction: get_current_weather: Helsinki\nPAUSE']:
    scripted = ScriptedAgent(response)
    rejected(query, 'test', 1, scripted)
    assert scripted.messages == []
print('PASS: real manual/automatic model-tool observations, two-argument forecast, conversions, strict parsing, turn bounds, isolated histories and atomic failure paths.')

## Original source-code and transcript appendix

Nonexecuting original archive material; use the maintained cells above.

```text
import openai
import re
import httpx
import os
from dotenv import load_dotenv

_ = load_dotenv()
from openai import OpenAI

client = OpenAI()
```

```text
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
                        model="gpt-4o", 
                        temperature=0,
                        messages=self.messages)
        return completion.choices[0].message.content
```

```text
prompt = """
You operate in a cycle of Thought, Action, PAUSE, and Observation.  
At the end of the loop, you provide an Answer.  

Use Thought to reason about the information needed to answer the question you were asked.  
Use Action to perform one of the available actions, then return PAUSE.  
Observation will be the result of performing those actions.  

Your available actions are:  

get_current_weather:  
e.g., get_current_weather: New York  
Fetches the current weather for the specified location.  

get_forecast:  
e.g., get_forecast: San Francisco, 3 days  
Provides a weather forecast for the given location and time range.  

convert_temperature:  
e.g., convert_temperature: 75 F to C  
Converts a temperature from Fahrenheit to Celsius or vice versa.  

Example session:  

Question: What’s the current temperature in Paris in Celsius?  
Thought: I need to get the current temperature in Paris and convert it to Celsius.  
Action: get_current_weather: Paris  
PAUSE  

You will then be called again with this:  

Observation: The current temperature in Paris is 77°F.  

You continue:  

Thought: I should convert 77°F to Celsius.  
Action: convert_temperature: 77 F to C  
PAUSE  

You will then be called again with this:  

Observation: 77°F is approximately 25°C.  

You finally output:  

Answer: The current temperature in Paris is 25°C.""".strip()
```

```text
def get_current_weather(location):
    # Placeholder implementation for fetching weather.
    return f"The current temperature in {location} is 32\u00b0F."

def get_forecast(location, days):
    # Placeholder implementation for fetching weather forecast.
    return f"The weather forecast for {location} over the next {days} days is sunny with mild temperatures."

def convert_temperature(conversion):
    try:
        value, unit_from, _, unit_to = conversion.split()
        value = float(value)

        if unit_from.upper() == "F" and unit_to.upper() == "C":
            converted = (value - 32) * 5 / 9
            return f"{value}\u00b0F is approximately {converted:.1f}\u00b0C."
        elif unit_from.upper() == "C" and unit_to.upper() == "F":
            converted = (value * 9 / 5) + 32
            return f"{value}\u00b0C is approximately {converted:.1f}\u00b0F."
        else:
            return "Invalid temperature conversion. Use 'F to C' or 'C to F'."
    except Exception as e:
        return f"Error in temperature conversion: {e}"

known_actions = {
    "get_current_weather": get_current_weather,
    "get_forecast": get_forecast,
    "convert_temperature": convert_temperature
}
```

```text
abot = Agent(prompt)
result = abot("What is the temperature in Helsinki?")
print(result)
```

```text
result = get_current_weather("Helsinki")
result
```

```text
result = get_forecast("Helsinki", 1)
result
```

```text
next_prompt = "Observation: {}".format(result)
abot(next_prompt)
```

```text
abot.messages
```

```text
action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action
def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [
            action_re.match(a) 
            for a in result.split('\n') 
            if action_re.match(a)
        ]
        if actions:
            # There is an action to run
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return
```

```text
Thought: I need to get the current temperature in Helsinki. 
Action: get_current_weather: Helsinki 
PAUSE
```

```text
‘The current temperature in Helsinki is 32°F.’
```

```text
‘’The weather forecast for Helsinki over the next 1 days is sunny with mild temperatures.’
```

```text
‘Answer: The weather in Helsinki is 0 C \nPAUSE’
```

```text
[{‘role’: ‘system’,
 ‘content’: “You operate in a cycle of Thought, Action, PAUSE, and Observation. \nAt the end of the loop, you provide an Answer. \n\nUse Thought to reason about the information needed to answer the question you were asked. \nUse Action to perform one of the available actions, then return PAUSE. \nObservation will be the result of performing those actions. \n\nYour available actions are: \n\nget_current_weather: \ne.g., get_current_weather: New York \nFetches the current weather for the specified location. \n\nget_forecast: \ne.g., get_forecast: San Francisco, 3 days \nProvides a weather forecast for the given location and time range. \n\nconvert_temperature: \ne.g., convert_temperature: 75 F to C \nConverts a temperature from Fahrenheit to Celsius or vice versa. \n\nExample session: \n\nQuestion: What’s the current temperature in Paris in Celsius? \nThought: I need to get the current temperature in Paris and convert it to Celsius. \nAction: get_current_weather: Paris \nPAUSE \n\nYou will then be called again with this: \n\nObservation: The current temperature in Paris is 77°F. \n\nYou continue: \n\nThought: I should convert 77°F to Celsius. \nAction: convert_temperature: 77 F to C \nPAUSE \n\nYou will then be called again with this: \n\nObservation: 77°F is approximately 25°C. \n\nYou finally output: \n\nAnswer: The current temperature in Paris is 25°C.”},
 {‘role’: ‘user’, ‘content’: ‘What is the temperature in Helsinki?’},
 {‘role’: ‘assistant’,
 ‘content’: ‘Thought: I need to get the current temperature in Helsinki. \nAction: get_current_weather: Helsinki \nPAUSE’},
 {‘role’: ‘user’,
 ‘content’: ‘Observation: The weather forecast for Helsinki over the next 1 days is sunny with mild temperatures.’},
 {‘role’: ‘assistant’,
 ‘content’: ‘Thought: It seems there was a misunderstanding. I need the current temperature, not the forecast. I will try to get the current temperature in Helsinki again. \nAction: get_current_weather: Helsinki \nPAUSE’},
 {‘role’: ‘assistant’,
 ‘content’: ‘Thought: The weather in Helsinki is 0 C \nPAUSE’}]
```

```text
Thought: I need to get the current temperature in Helsinki. 
Action: get_current_weather: Helsinki 
PAUSE
 — running get_current_weather Helsinki 
Observation: The current temperature in Helsinki is 32°F.
Thought: I should convert 32°F to Celsius to provide the temperature in a more commonly used unit in Helsinki. 
Action: convert_temperature: 32 F to C 
PAUSE
 — running convert_temperature 32F to C 
Observation: 32.0°F is approximately 0°C.
Answer: The current temperature in Helsinki is approximately 0°C.
```